# Семинар 2. Фильтры, границы, морфология

Добавляем шум, сглаживаем, ищем границы, выделяем объекты порогом и чистим маску морфологией. В конце находим на двух снимках одни и те же точки SIFT и склеиваем снимки в панораму. Для каждого шага смотрим, что он улучшил и что потерял. Большая часть опытов на синтетической сцене, где известен правильный ответ, остальное на фотографии бабочки из первого семинара.

Нужны `numpy`, `matplotlib`, `opencv-python`. Внешние файлы не нужны.

In [ ]:
import base64

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)
print(np.__version__, cv.__version__)

Картинка в base64, ячейка свёрнута.

In [ ]:
BUTTERFLY_JPEG_B64 = (
    "..."
    "2Q=="
)

In [ ]:
photo = cv.cvtColor(cv.imdecode(np.frombuffer(base64.b64decode(BUTTERFLY_JPEG_B64), np.uint8), cv.IMREAD_COLOR), cv.COLOR_BGR2RGB)
photo_gray = cv.cvtColor(photo, cv.COLOR_RGB2GRAY)
print(photo.shape, photo_gray.shape)

## Сцена с известным ответом

Три объекта на фоне, яркость которого плавно растёт слева направо. Истинная маска нужна, чтобы проверять результат числом, а не на глаз.

In [ ]:
H, W = 240, 320
truth_u8 = np.zeros((H, W), np.uint8)
cv.rectangle(truth_u8, (25, 45), (85, 185), 255, -1)
cv.circle(truth_u8, (155, 110), 35, 255, -1)
cv.ellipse(truth_u8, (255, 125), (28, 60), 0, 0, 360, 255, -1)
truth = truth_u8 > 0

illumination = np.linspace(0, 80, W)[None, :]
clean = np.where(truth, 165 + illumination, 35 + illumination).astype(np.float32)
noisy = np.clip(clean + rng.normal(0, 16, (H, W)), 0, 255).astype(np.uint8)

row = 110
fig, axes = plt.subplots(1, 4, figsize=(17, 3), gridspec_kw={"width_ratios": [1, 1, 1, 1.6]})
for ax, im, title in zip(axes, [clean, noisy, truth], ["без шума", "гауссов шум, sigma = 16", "истинная маска"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255 if im.dtype != bool else 1)
    ax.axhline(row, c="red", lw=0.7)
    ax.set_title(title)
    ax.axis("off")
axes[3].plot(noisy[row], lw=0.7, label="noisy")
axes[3].plot(clean[row], lw=2, label="clean")
axes[3].set_title(f"строка y = {row}")
axes[3].legend()
plt.show()

**Что видно:** на профиле строки объекты это ступеньки, а фон и сами объекты плавно светлеют слева направо. Шум с sigma 16 сравним с половиной этого перепада освещения. Маска от освещения не зависит: пиксель либо объект, либо нет.

Для сравнения масок используем IoU: площадь пересечения, делённая на площадь объединения.

In [ ]:
def mask_iou(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 1.0

print(mask_iou(truth, truth), mask_iou(np.zeros_like(truth), truth))

## Свёртка руками

Каждый выходной пиксель это взвешенная сумма соседей. `filter2D` считает корреляцию, то есть ядро не переворачивает. Для симметричных ядер это то же самое, что свёртка, для Собеля знак будет зависеть от этого.

In [ ]:
kernel = np.ones((3, 3), np.float32) / 9
average = cv.filter2D(noisy.astype(np.float32), -1, kernel)

y, x = 100, 50
patch = noisy[y - 1 : y + 2, x - 1 : x + 2]
print("руками:", (patch * kernel).sum(), "filter2D:", average[y, x])

fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, m, title in zip(axes, [patch, kernel, None], ["окно 3x3", "ядро", "результат"]):
    if m is None:
        ax.imshow([[average[y, x]]], cmap="gray", vmin=0, vmax=255)
        ax.text(0, 0, f"{average[y, x]:.1f}", ha="center", va="center", color="red", fontsize=14)
    else:
        ax.imshow(m, cmap="gray", vmin=0, vmax=255 if m is patch else m.max() * 2)
        for (i, j), v in np.ndenumerate(m):
            ax.text(j, i, f"{v:.2f}" if m is kernel else v, ha="center", va="center", color="red")
    ax.set_title(title)
    ax.axis("off")
plt.show()

На краю у пикселя нет части соседей, и их надо чем-то заполнить. Способ заполнения задаётся `borderType`. Посмотрим, как выглядят разные режимы, если расширить фотографию на 60 пикселей.

In [ ]:
modes = {
    "CONSTANT": cv.BORDER_CONSTANT,
    "REPLICATE": cv.BORDER_REPLICATE,
    "REFLECT_101": cv.BORDER_REFLECT_101,
    "WRAP": cv.BORDER_WRAP,
}
small = cv.resize(photo, (200, 133), interpolation=cv.INTER_AREA)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, (name, mode) in zip(axes, modes.items()):
    padded = cv.copyMakeBorder(small, 60, 60, 60, 60, mode)
    ax.imshow(padded)
    ax.add_patch(plt.Rectangle((60, 60), 200, 133, fill=False, ec="red"))
    ax.set_title(name)
    ax.axis("off")
plt.show()

**Что видно:** CONSTANT добавляет чёрную рамку, REPLICATE тянет крайние пиксели полосами, REFLECT_101 отражает картинку как в зеркале, WRAP подставляет противоположный край. По умолчанию в OpenCV REFLECT_101.

Чёрная рамка это резкий перепад, и любой фильтр границ его найдёт. Проверим на постоянной картинке, где границ нет вообще.

In [ ]:
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
sobel_y = sobel_x.T

constant = np.full((40, 60), 100, np.float32)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, mode, name in zip(axes, [cv.BORDER_CONSTANT, cv.BORDER_REFLECT_101], ["CONSTANT", "REFLECT_101"]):
    g = cv.filter2D(constant, -1, sobel_x, borderType=mode)
    ax.imshow(g, cmap="coolwarm", vmin=-400, vmax=400)
    ax.set_title(f"{name}, max |Gx| = {np.abs(g).max():.0f}")
    ax.axis("off")
plt.show()

**Что видно:** с нулевым заполнением у левого и правого края появились сильные ответы, хотя картинка постоянная. С отражением ответ ноль везде. Граница кадра не должна превращаться в границу объекта.

Операция одна, а результат целиком определяется ядром. Пять ядер на одной фотографии: сверху ядро, снизу результат. Чтобы ядро было видно, рисуем его в своём масштабе: красное положительные веса, синее отрицательные.

In [ ]:
g32 = photo_gray.astype(np.float32)
gk = cv.getGaussianKernel(9, 2)
kernels = {
    "среднее 9x9": np.ones((9, 9), np.float32) / 81,
    "Gaussian, sigma 2": (gk @ gk.T).astype(np.float32),
    "резкость": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32),
    "Собель по x": sobel_x,
    "Лаплас": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32),
}
fig, axes = plt.subplots(2, 6, figsize=(22, 6), gridspec_kw={"height_ratios": [1, 2]})
axes[0, 0].axis("off")
axes[1, 0].imshow(photo_gray, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("исходная")
for j, (name, k) in enumerate(kernels.items(), start=1):
    lim = np.abs(k).max()
    axes[0, j].imshow(k, cmap="coolwarm", vmin=-lim, vmax=lim)
    if k.shape[0] <= 3:
        for (i, jj), v in np.ndenumerate(k):
            axes[0, j].text(jj, i, f"{v:g}", ha="center", va="center", fontsize=9)
    axes[0, j].set_title(name)
    out = cv.filter2D(g32, -1, k)
    if k.sum() == 0:
        lim2 = np.percentile(np.abs(out), 99)
        axes[1, j].imshow(out, cmap="coolwarm", vmin=-lim2, vmax=lim2)
    else:
        axes[1, j].imshow(np.clip(out, 0, 255), cmap="gray", vmin=0, vmax=255)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

**Что видно:** у ядер с суммой весов 1 (среднее, Gaussian, резкость) средняя яркость сохраняется, и результат похож на картинку: размытую или более резкую. У ядер с суммой 0 (Собель, Лаплас) на ровных участках ответ ноль, а остаются только перепады, поэтому результат рисуем красно-синим. Резкость это картинка плюс минус Лаплас: исходный пиксель с весом 5 минус четыре соседа.

## Сглаживание

Три фильтра с окном 5: среднее, гауссово и медианное. Ошибку считаем относительно чистой сцены.

In [ ]:
mean5 = cv.blur(noisy, (5, 5))
gauss = cv.GaussianBlur(noisy, (5, 5), 1.2)
median = cv.medianBlur(noisy, 5)
filters = {"noisy": noisy, "mean": mean5, "Gaussian": gauss, "median": median}
mse = {name: float(np.mean((im.astype(float) - clean) ** 2)) for name, im in filters.items()}
print({k: round(v, 1) for k, v in mse.items()})

fig, axes = plt.subplots(1, 6, figsize=(21, 3), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1, 1.4]})
axes[0].imshow(clean[60:160, 100:210], cmap="gray", vmin=0, vmax=255)
axes[0].set_title("чистая сцена")
for ax, (name, im) in zip(axes[1:], filters.items()):
    ax.imshow(im[60:160, 100:210], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{name}, MSE {mse[name]:.0f}")
for ax in axes[:5]:
    ax.axis("off")
for name, im in filters.items():
    axes[5].plot(im[row, 100:210], lw=1 if name == "noisy" else 1.5, label=name, alpha=0.5 if name == "noisy" else 1)
axes[5].plot(clean[row, 100:210], "k--", lw=1, label="clean")
axes[5].set_title("профиль через круг")
axes[5].legend(fontsize=8)
plt.show()

**Что видно:** все три фильтра снижают ошибку в несколько раз, а медиана сильнее всех, MSE 27 против 65 у гаусса. На профиле все три идут за ступенькой, но у среднего и гаусса край чуть наклоннее: усреднение смешивает пиксели по обе стороны границы. Медиана выбирает одно из значений окна и край не размывает.

Теперь гаусс с разной sigma на фотографии.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, s in zip(axes, [0, 1, 3, 8]):
    im = photo if s == 0 else cv.GaussianBlur(photo, (0, 0), s)
    ax.imshow(im)
    ax.set_title("исходное" if s == 0 else f"sigma = {s}")
    ax.axis("off")
plt.show()

**Что видно:** при sigma 1 пропадает в основном шум JPEG, при 3 уже теряются белые точки на крыльях, при 8 остаются только крупные цветовые пятна. Размер деталей, которые переживают сглаживание, порядка sigma.

## Импульсный шум

Другой тип шума: редкие пиксели заменены на чёрный или белый. Отдельно считаем ошибку внутри объектов, вдали от границ.

In [ ]:
impulse = clean.astype(np.uint8).copy()
u = rng.random((H, W))
impulse[u < 0.025] = 0
impulse[u > 0.975] = 255

interior = cv.erode(truth_u8, np.ones((11, 11), np.uint8)) > 0
results = {"impulse": impulse, "Gaussian": cv.GaussianBlur(impulse, (5, 5), 1.2), "median": cv.medianBlur(impulse, 5)}
for name, im in results.items():
    err = (im.astype(float) - clean) ** 2
    print(f"{name:9s} MSE всего {err.mean():7.1f}  внутри объектов {err[interior].mean():7.1f}")

fig, axes = plt.subplots(1, 4, figsize=(17, 3))
for ax, (name, im) in zip(axes, [("чистая сцена", clean)] + list(results.items())):
    ax.imshow(im[60:160, 100:210], cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis("off")
plt.show()

**Что видно:** гаусс не убирает выбросы, а размазывает каждый в пятно 5 на 5. Медиана почти полностью их удаляет: один выброс в окне из 25 пикселей не может стать медианой. Внутри объектов у медианы ошибка почти ноль.

## Производные

Ядро Собеля это разность соседей, сглаженная по перпендикулярному направлению. Считаем во float32, чтобы отрицательные производные не обрезались. Сначала проверим на рампе, где ответ известен заранее: яркость растёт на 1 за пиксель.

In [ ]:
ramp = np.tile(np.arange(100, dtype=np.float32), (80, 1))
rx = cv.filter2D(ramp, -1, sobel_x)
ry = cv.filter2D(ramp, -1, sobel_y)
print("Gx внутри:", np.unique(rx[2:-2, 2:-2]), "Gy внутри:", np.unique(ry[2:-2, 2:-2]))

Gx равен 8: разность через пиксель даёт 2, а веса 1, 2, 1 в сумме дают 4. Gy ноль, потому что по вертикали ничего не меняется.

In [ ]:
gx = cv.filter2D(gauss.astype(np.float32), -1, sobel_x)
gy = cv.filter2D(gauss.astype(np.float32), -1, sobel_y)
magnitude = np.hypot(gx, gy)
angle = np.arctan2(gy, gx)

def angle_image(mag, ang):
    hsv = np.zeros(mag.shape + (3,), np.uint8)
    hsv[..., 0] = ((ang + np.pi) / (2 * np.pi) * 179).astype(np.uint8)
    hsv[..., 1] = 255
    hsv[..., 2] = np.clip(mag / np.percentile(mag, 99) * 255, 0, 255).astype(np.uint8)
    return cv.cvtColor(hsv, cv.COLOR_HSV2RGB)

lim = np.abs(gx).max()
fig, axes = plt.subplots(1, 5, figsize=(21, 3))
axes[0].imshow(gauss, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("вход: сцена после Gaussian")
axes[1].imshow(gx, cmap="coolwarm", vmin=-lim, vmax=lim)
axes[1].set_title("Gx")
axes[2].imshow(gy, cmap="coolwarm", vmin=-lim, vmax=lim)
axes[2].set_title("Gy")
axes[3].imshow(magnitude, cmap="gray")
axes[3].set_title("модуль")
axes[4].imshow(angle_image(magnitude, angle))
axes[4].set_title("направление цветом")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** Gx положительный (красный) на левых краях объектов, где слева тёмный фон, а справа светлый объект, и отрицательный (синий) на правых. Gy так же для верхних и нижних краёв. Градиент направлен от тёмного к светлому, то есть внутрь объекта, поэтому на карте направлений цвет обходит круг и эллипс по всей палитре. Плавный рост освещения в модуль почти не попадает: 80 уровней на 320 пикселей это в десятки раз слабее края.

Если считать в uint8, отрицательная половина пропадает.

In [ ]:
signed = cv.filter2D(truth_u8, cv.CV_32F, sobel_x)
clipped = cv.filter2D(truth_u8, cv.CV_8U, sobel_x)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].imshow(truth_u8, cmap="gray")
axes[0].set_title("вход: маска")
axes[1].imshow(signed, cmap="coolwarm", vmin=-1020, vmax=1020)
axes[1].set_title(f"Gx во float32: от {signed.min():.0f} до {signed.max():.0f}")
axes[2].imshow(clipped, cmap="gray")
axes[2].set_title(f"Gx в uint8: от {clipped.min()} до {clipped.max()}")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** в uint8 остались только левые края объектов. Правые края дали отрицательный ответ, и он обрезался до нуля. Если после этого искать границы, половина их потеряется молча.

То же на фотографии.

In [ ]:
pg = cv.GaussianBlur(photo_gray, (0, 0), 1).astype(np.float32)
pgx = cv.Sobel(pg, cv.CV_32F, 1, 0)
pgy = cv.Sobel(pg, cv.CV_32F, 0, 1)
pmag = np.hypot(pgx, pgy)
fig, axes = plt.subplots(1, 3, figsize=(19, 4.3))
axes[0].imshow(photo)
axes[0].set_title("исходная фотография")
axes[1].imshow(pmag, cmap="gray", vmax=np.percentile(pmag, 99))
axes[1].set_title("модуль градиента")
axes[2].imshow(angle_image(pmag, np.arctan2(pgy, pgx)))
axes[2].set_title("направление")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** сильнее всего отвечают жилки и белые точки, у них самый резкий перепад. Каждая жилка на карте направлений двухцветная: с одной стороны градиент смотрит в одну сторону, с другой в противоположную.

## Canny

Canny берёт модуль градиента, оставляет только локальные максимумы поперёк границы и применяет два порога: всё выше верхнего это граница, всё между порогами остаётся, только если связано с сильной границей.

In [ ]:
edges_raw = cv.Canny(noisy, 60, 120)
edges_smooth = cv.Canny(gauss, 60, 120)
fig, axes = plt.subplots(1, 4, figsize=(19, 3.2))
axes[0].imshow(noisy, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("вход: шумная сцена")
axes[1].imshow(edges_raw, cmap="gray")
axes[1].set_title(f"Canny, {np.count_nonzero(edges_raw)} пикселей")
axes[2].imshow(gauss, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("вход: после Gaussian")
axes[3].imshow(edges_smooth, cmap="gray")
axes[3].set_title(f"Canny, {np.count_nonzero(edges_smooth)} пикселей")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** на шумной картинке Canny находит границы почти везде, шум с sigma 16 даёт перепады выше порога. После гаусса остаются контуры трёх объектов толщиной в пиксель. Карта границ это не маска: внутри объектов она пустая.

In [ ]:
pairs = [(20, 60), (50, 150), (100, 250)]
fig, axes = plt.subplots(1, 4, figsize=(21, 3.8))
axes[0].imshow(photo)
axes[0].set_title("исходная фотография")
axes[0].axis("off")
for ax, (lo, hi) in zip(axes[1:], pairs):
    e = cv.Canny(cv.GaussianBlur(photo_gray, (0, 0), 1.4), lo, hi)
    ax.imshow(e, cmap="gray")
    ax.set_title(f"пороги {lo}, {hi}: {np.count_nonzero(e)} пикселей")
    ax.axis("off")
plt.show()

**Что видно:** низкие пороги ловят текстуру листьев, высокие оставляют только жилки и контур крыла. Хорошей пары порогов для всех картинок нет, её подбирают под задачу.

Разберём Canny по шагам на куске фотографии. Сначала модуль градиента. Потом подавление немаксимумов: пиксель остаётся, только если он больше двух соседей вдоль направления градиента, то есть поперёк границы. Толстый край становится линией в один пиксель. Потом два порога: сильные пиксели выше верхнего (белые), слабые между порогами (жёлтые). В конце остаются слабые, связанные с сильными.

In [ ]:
crop = cv.GaussianBlur(photo_gray, (0, 0), 1.4)[120:300, 180:420].astype(np.float32)
cgx = cv.Sobel(crop, cv.CV_32F, 1, 0)
cgy = cv.Sobel(crop, cv.CV_32F, 0, 1)
cmag = np.hypot(cgx, cgy)
direction = (np.round(np.arctan2(cgy, cgx) / (np.pi / 4)) % 4).astype(int)
offsets = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (1, -1)}
nms = np.zeros_like(cmag)
for d, (dy, dx) in offsets.items():
    fwd = np.roll(cmag, (-dy, -dx), axis=(0, 1))
    bwd = np.roll(cmag, (dy, dx), axis=(0, 1))
    keep = (direction == d) & (cmag >= fwd) & (cmag >= bwd)
    nms[keep] = cmag[keep]
lo, hi = 50, 150
strong, weak = nms >= hi, (nms >= lo) & (nms < hi)
classes = np.zeros(crop.shape + (3,), np.uint8)
classes[weak] = (230, 200, 40)
classes[strong] = 255
final = cv.Canny(crop.astype(np.uint8), 50, 150)

fig, axes = plt.subplots(1, 5, figsize=(22, 3.6))
for ax, im, title in zip(axes, [crop, cmag, nms, classes, final],
                         ["кусок фотографии", "модуль градиента", "после подавления немаксимумов", "сильные и слабые", "итог Canny"]):
    ax.imshow(im, cmap="gray", vmax=None if im.ndim == 3 or im is crop or im is final else np.percentile(cmag, 99))
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()
print("сильных", strong.sum(), "слабых", weak.sum(), "в итоге", np.count_nonzero(final))

**Что видно:** модуль градиента даёт полосы шириной в несколько пикселей. После подавления немаксимумов от каждой полосы остаётся гребень в один пиксель. Порог по одному верхнему значению оставил бы рваные куски жилок. Нижний порог добавляет слабые пиксели, но только те, что продолжают сильную линию, и жилки становятся непрерывными, а слабые пиксели в текстуре листьев отбрасываются. Наша схема упрощённая, направление округляем до четырёх, но на глаз итог совпадает с `cv.Canny`.

## Пороги

Фиксированный порог сравнивает все пиксели с одной константой. Otsu выбирает порог по гистограмме так, чтобы два класса были как можно компактнее. Адаптивный порог сравнивает пиксель со средним по окрестности.

In [ ]:
_, fixed = cv.threshold(gauss, 140, 255, cv.THRESH_BINARY)
otsu_t, otsu = cv.threshold(gauss, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
adaptive = cv.adaptiveThreshold(gauss, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, cv.THRESH_BINARY, 41, -7)
masks = {"fixed 140": fixed, f"Otsu {otsu_t:.0f}": otsu, "adaptive": adaptive}

def error_map(pred, truth):
    pred = pred > 0
    out = np.zeros(truth.shape + (3,), np.uint8)
    out[pred & truth] = (230, 230, 230)
    out[pred & ~truth] = (230, 60, 60)
    out[~pred & truth] = (60, 120, 230)
    return out

fig, axes = plt.subplots(1, 5, figsize=(21, 3), gridspec_kw={"width_ratios": [1, 1, 1, 1, 1.4]})
axes[0].imshow(gauss, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("вход: сцена после Gaussian")
axes[0].axis("off")
for ax, (name, m) in zip(axes[1:], masks.items()):
    ax.imshow(error_map(m, truth))
    ax.set_title(f"{name}, IoU {mask_iou(m, truth):.3f}")
    ax.axis("off")
axes[4].hist(gauss[truth].ravel(), bins=60, alpha=0.5, label="объект")
axes[4].hist(gauss[~truth].ravel(), bins=60, alpha=0.5, label="фон")
axes[4].axvline(otsu_t, c="k", label="Otsu")
axes[4].axvline(140, c="k", ls="--", label="140")
axes[4].legend()
plt.show()

Белое это верно найденный объект, красное лишнее, синее пропущенное.

**Что видно:** после гаусса распределения фона и объекта почти не перекрываются, и оба глобальных порога дают IoU около 0.98. Ошибки только по краю: красная кайма у эллипса справа, где фон светлее всего, и синие пиксели у прямоугольника слева, где объект темнее всего. Otsu выбрал 136, это почти тот же порог, что 140.

Адаптивный порог провалился: IoU 0.47. Он сравнивает пиксель со средним по окну 41 на 41, а внутри объекта крупнее окна пиксель и есть среднее, поэтому середины объектов пропали, остались только полосы у краёв. Плюс шумные точки в фоне. Адаптивный порог хорош для мелких структур на неровном фоне, например текста, к нему вернёмся на семинаре про документы.

Как Otsu выбирает порог. Для каждого порога $t$ делим пиксели на два класса и считаем межклассовую дисперсию $w_0 w_1 (\mu_0 - \mu_1)^2$, где $w$ это доли классов, а $\mu$ их средние. Чем дальше средние друг от друга и чем ровнее делятся пиксели, тем больше. Otsu берёт максимум.

In [ ]:
hist = np.bincount(gauss.ravel(), minlength=256).astype(float)
prob = hist / hist.sum()
levels = np.arange(256)
w0 = np.cumsum(prob)
mu0_sum = np.cumsum(prob * levels)
w1 = 1 - w0
mu0 = mu0_sum / np.maximum(w0, 1e-12)
mu1 = (mu0_sum[-1] - mu0_sum) / np.maximum(w1, 1e-12)
between = w0 * w1 * (mu0 - mu1) ** 2
print("максимум межклассовой дисперсии на", between.argmax(), " cv.threshold выбрал", otsu_t)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(levels, hist, width=1, color="lightgray", label="гистограмма")
ax2 = ax.twinx()
ax2.plot(levels, between, c="C3", label="межклассовая дисперсия")
ax2.axvline(between.argmax(), c="C3", ls="--")
ax.set_xlabel("яркость")
ax.set_ylabel("пикселей")
ax2.set_ylabel("дисперсия")
fig.legend(loc="upper left", bbox_to_anchor=(0.13, 0.87))
plt.show()

**Что видно:** гистограмма из двух горбов, фон и объекты. У межклассовой дисперсии широкое плато во впадине между горбами, максимум на 136, и наш расчёт совпал с `cv.threshold`. Если горбы сливаются, максимум становится пологим, и Otsu теряет уверенность: так будет на тёмной части листа в семинаре про документы.

Затемним картинку на 50 уровней. Маска объектов та же.

In [ ]:
dark = np.clip(gauss.astype(np.int16) - 50, 0, 255).astype(np.uint8)
_, dark_fixed = cv.threshold(dark, 140, 255, cv.THRESH_BINARY)
dark_t, dark_otsu = cv.threshold(dark, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
axes[0].imshow(dark, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("на 50 темнее")
axes[1].imshow(error_map(dark_fixed, truth))
axes[1].set_title(f"fixed 140, IoU {mask_iou(dark_fixed, truth):.3f}")
axes[2].imshow(error_map(dark_otsu, truth))
axes[2].set_title(f"Otsu {dark_t:.0f}, IoU {mask_iou(dark_otsu, truth):.3f}")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** фиксированный порог потерял весь левый прямоугольник и край круга: они стали темнее 140, IoU упал до 0.5. Otsu сдвинул порог со 136 до 87, почти на те же 50 уровней, и дал прежние 0.98. Порог, подобранный руками на одной съёмке, ломается при другом свете.

Теперь будем двигать порог и смотреть на два вида ошибок.

In [ ]:
ts = np.arange(60, 250, 5)
rows = []
for t in ts:
    pred = gauss > t
    tp = (pred & truth).sum()
    fp = (pred & ~truth).sum()
    fn = (~pred & truth).sum()
    rows.append((tp / max(tp + fp, 1), tp / max(tp + fn, 1), mask_iou(pred, truth)))
precision, recall, iou = np.array(rows).T
best = ts[iou.argmax()]
print("лучший порог по IoU:", best, "IoU", iou.max().round(3), "Otsu:", otsu_t)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(ts, precision, label="precision")
axes[0].plot(ts, recall, label="recall")
axes[0].plot(ts, iou, label="IoU")
axes[0].axvline(otsu_t, c="k", ls="--", lw=0.8)
axes[0].set_xlabel("порог")
axes[0].legend()
axes[1].plot(recall, precision, "o-", ms=3)
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
plt.show()

**Что видно:** при низком пороге recall единица, но в маску попадает фон, precision падает. При высоком наоборот. IoU максимален на пороге 125, и Otsu со своими 136 попадает близко к этому месту, хотя ничего о маске не знает. На такой простой сцене кривая precision от recall почти прямой угол. Кривая precision от recall это та же информация без оси порога, к ней вернёмся на семинаре про детекцию.

## Морфология

Erosion оставляет пиксель, только если весь структурный элемент вокруг него внутри маски, и маска сжимается. Dilation наоборот расширяет. Opening это erosion, потом dilation: убирает мелкие светлые пятна. Closing это dilation, потом erosion: закрывает мелкие тёмные дырки.

In [ ]:
demo = np.zeros((60, 100), np.uint8)
cv.rectangle(demo, (15, 15), (45, 45), 255, -1)
cv.circle(demo, (75, 30), 15, 255, -1)
demo[28:32, 25:35] = 0
demo[5, 60] = demo[50, 8] = demo[52, 90] = 255
el = cv.getStructuringElement(cv.MORPH_ELLIPSE, (5, 5))
ops = {
    "исходная": demo,
    "erode": cv.erode(demo, el),
    "dilate": cv.dilate(demo, el),
    "open": cv.morphologyEx(demo, cv.MORPH_OPEN, el),
    "close": cv.morphologyEx(demo, cv.MORPH_CLOSE, el),
}
fig, axes = plt.subplots(1, 5, figsize=(18, 2.6))
for ax, (name, m) in zip(axes, ops.items()):
    ax.imshow(m, cmap="gray")
    ax.contour(demo, [127], colors="red", linewidths=0.7)
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

Красный контур это исходная маска.

**Что видно:** erode сжимает объекты, убирает отдельные точки и расширяет дырку в квадрате. Dilate наоборот закрывает дырку, но и точки раздувает в круги. Open убирает точки и сохраняет форму, но дырка остаётся. Close закрывает дырку, но точки остаются.

In [ ]:
damaged = truth_u8.copy()
damaged[rng.integers(0, H, 500), rng.integers(0, W, 500)] = 255
for cx, cy in [(155, 110), (55, 100), (255, 140)]:
    cv.circle(damaged, (cx, cy), 2, 0, -1)
opened = cv.morphologyEx(damaged, cv.MORPH_OPEN, el)
cleaned = cv.morphologyEx(opened, cv.MORPH_CLOSE, el)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, m, title in zip(axes, [damaged, opened, cleaned], ["повреждена", "open", "open + close"]):
    ax.imshow(error_map(m, truth))
    ax.set_title(f"{title}, IoU {mask_iou(m, truth):.3f}")
    ax.axis("off")
plt.show()

**Что видно:** красные это 500 случайных точек, синие три маленькие дырки. Open убрал точки, IoU вырос с 0.977 до 0.997. Close закрыл дырки, 0.999. Не ровно единица: эллиптический элемент немного скругляет острые углы прямоугольника.

Найдём связные компоненты и подпишем каждую.

In [ ]:
n, labels, stats, centroids = cv.connectedComponentsWithStats(cleaned, connectivity=8)
print("компонент без фона:", n - 1)

vis = cv.cvtColor(noisy, cv.COLOR_GRAY2RGB)
colors = plt.cm.tab10(np.arange(n))[:, :3]
overlay = vis.copy()
for k in range(1, n):
    overlay[labels == k] = (colors[k] * 255).astype(np.uint8)
vis = cv.addWeighted(vis, 0.5, overlay, 0.5, 0)
for k in range(1, n):
    x, y, w_, h_, area = stats[k]
    cv.rectangle(vis, (x, y), (x + w_ - 1, y + h_ - 1), (255, 255, 0), 1)
    cv.putText(vis, f"{area}", (x, y - 4), cv.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
plt.figure(figsize=(7, 5))
plt.imshow(vis)
plt.axis("off")
plt.show()

Морфология умеет и испортить результат. Два круга с зазором в несколько пикселей, closing с разным размером элемента.

In [ ]:
two = np.zeros((80, 120), np.uint8)
cv.circle(two, (39, 40), 20, 255, -1)
cv.circle(two, (82, 40), 20, 255, -1)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, k in zip(axes, [3, 7, 15]):
    m = cv.morphologyEx(two, cv.MORPH_CLOSE, cv.getStructuringElement(cv.MORPH_ELLIPSE, (k, k)))
    count = cv.connectedComponents(m)[0] - 1
    ax.imshow(m, cmap="gray")
    ax.set_title(f"элемент {k}: объектов {count}")
    ax.axis("off")
plt.show()

**Что видно:** при элементе 3 круги остаются раздельными, при 7 и больше между ними появляется перемычка, и это уже один объект. IoU такую перемычку почти не заметит, площадь изменилась мало, а число объектов изменилось вдвое.

Последнее: что считать соседями. Диагональная линия из 7 пикселей.

In [ ]:
diagonal = np.eye(7, dtype=np.uint8) * 255
counts = {c: cv.connectedComponents(diagonal, connectivity=c)[0] - 1 for c in (4, 8)}
print(counts)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(diagonal, cmap="gray")
axes[0].set_title("маска")
for ax, c in zip(axes[1:], (4, 8)):
    ax.imshow(cv.connectedComponents(diagonal, connectivity=c)[1], cmap="tab10", vmin=0, vmax=9)
    ax.set_title(f"{c}-связность: {counts[c]}")
for ax in axes:
    ax.set_xticks(np.arange(-0.5, 7))
    ax.set_yticks(np.arange(-0.5, 7))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(c="gray", lw=0.5)
plt.show()

**Что видно:** при 4-связности соседи только по стороне, и каждый пиксель диагонали отдельный объект. При 8-связности соседи и по углу, и это одна линия.

## Особые точки: SIFT и склейка панорамы

Градиенты нужны не только для границ. Если найти на двух снимках одни и те же точки, можно понять, как снимки связаны, и склеить их. Так работает панорама в телефоне.

Какие точки годятся? Сдвинем маленькое окно и посмотрим, насколько изменилось его содержимое: сумма квадратов разностей, SSD, для всех сдвигов от -6 до 6 пикселей. Три окна: ровный участок, край и пятно. Цветовая шкала у трёх карт общая.

In [ ]:
g = photo_gray.astype(np.float32)
spots = {"ровно": (48, 416), "край": (302, 330), "пятно": (212, 148)}
def ssd_surface(y, x):
    win = g[y - 7 : y + 8, x - 7 : x + 8]
    return np.array([[((g[y - 7 + dy : y + 8 + dy, x - 7 + dx : x + 8 + dx] - win) ** 2).sum()
                      for dx in range(-6, 7)] for dy in range(-6, 7)])

surfaces = {name: ssd_surface(y, x) for name, (y, x) in spots.items()}
vmax = max(v.max() for v in surfaces.values())
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for j, (name, (y, x)) in enumerate(spots.items()):
    ssd = surfaces[name]
    axes[0, j].imshow(photo[y - 30 : y + 31, x - 30 : x + 31])
    axes[0, j].add_patch(plt.Rectangle((22.5, 22.5), 15, 15, fill=False, ec="cyan", lw=2))
    axes[0, j].set_title(f"{name}: окно 15 на 15")
    axes[0, j].axis("off")
    axes[1, j].imshow(ssd, cmap="viridis", vmin=0, vmax=vmax, extent=(-6.5, 6.5, 6.5, -6.5))
    axes[1, j].set_title(f"SSD при сдвиге, максимум {ssd.max():.0f}")
    axes[1, j].set_xlabel("dx")
plt.tight_layout()
plt.show()

Окна выбраны по градиентам внутри них: где градиента нет, где он сильный в одном направлении и где сильный в двух.

**Что видно:** на ровном участке SSD почти не меняется ни при каком сдвиге: окно можно сдвинуть куда угодно, и мы этого не заметим. На краю SSD растёт поперёк края, а вдоль него остаётся долина: положение вдоль края не определено. В третьем окне белое пятно на тёмном, и у его краёв градиент идёт во всех направлениях, как у угла: SSD растёт во все стороны, и минимум единственный. Шкала общая, поэтому на ровном участке карта целиком тёмная: SSD там в сотни раз меньше. Именно такие точки и можно найти на другом снимке. Детектор Харриса проверяет это через градиенты в окне, не перебирая сдвиги.

Теперь два снимка. Делаем их из бабочки, как будто камеру повернули: левый кадр это левая часть фотографии, правый это правая часть, повёрнутая, увеличенная, с лёгкой перспективой и на 30% светлее, как будто камера заново выставила экспозицию. Гомография между кадрами известна, и каждое найденное соответствие можно проверить.

In [ ]:
FW = photo.shape[1] * 5 // 8
Hp, Wp = photo.shape[:2]
ang, sc = np.deg2rad(6), 1.08
to_center = np.array([[1, 0, -0.62 * Wp], [0, 1, -Hp / 2], [0, 0, 1]])
rot = np.array([[sc * np.cos(ang), -sc * np.sin(ang), 0], [sc * np.sin(ang), sc * np.cos(ang), 0], [0, 0, 1]])
persp = np.array([[1, 0, 0], [0, 1, 0], [0.00025, 0, 1]])
from_center = np.array([[1, 0, FW / 2], [0, 1, Hp / 2], [0, 0, 1]])
H_photo_to_B = from_center @ persp @ rot @ to_center

frame_a = photo[:, :FW].copy()
frame_b = cv.warpPerspective(photo, H_photo_to_B, (FW, Hp))
frame_b = np.clip(frame_b.astype(np.float32) * 1.3 + 5, 0, 255).astype(np.uint8)
H_true = np.linalg.inv(H_photo_to_B)
H_true /= H_true[2, 2]

corners_b = np.float32([[0, 0], [FW, 0], [FW, Hp], [0, Hp]])[None]
true_corners = cv.perspectiveTransform(corners_b, H_true)[0]
fig, axes = plt.subplots(1, 3, figsize=(19, 3.8), gridspec_kw={"width_ratios": [1.6, 1, 1]})
axes[0].imshow(photo)
axes[0].add_patch(plt.Rectangle((0, 0), FW, Hp, fill=False, ec="C0", lw=2, label="кадр A"))
axes[0].add_patch(plt.Polygon(true_corners, fill=False, ec="C1", lw=2, label="кадр B"))
axes[0].legend(loc="lower left")
axes[0].set_title("исходная фотография и два кадра")
axes[1].imshow(frame_a)
axes[1].set_title("кадр A")
axes[2].imshow(frame_b)
axes[2].set_title("кадр B")
for ax in axes:
    ax.axis("off")
plt.show()

SIFT находит точки на разных масштабах: размывает картинку гауссом с растущей sigma и ищет экстремумы разности соседних размытий. У каждой точки есть масштаб и направление, по преобладающему направлению градиента вокруг неё. Дескриптор это гистограммы направлений градиента в сетке 4 на 4 вокруг точки, по 8 направлений, всего 128 чисел. Окно поворачивается вместе с направлением точки и растёт с её масштабом, поэтому дескриптор не меняется при повороте и увеличении.

In [ ]:
sift = cv.SIFT_create()
gray_a = cv.cvtColor(frame_a, cv.COLOR_RGB2GRAY)
gray_b = cv.cvtColor(frame_b, cv.COLOR_RGB2GRAY)
kp_a, des_a = sift.detectAndCompute(gray_a, None)
kp_b, des_b = sift.detectAndCompute(gray_b, None)
print("точек в A:", len(kp_a), " в B:", len(kp_b), " дескриптор:", des_a.shape[1])

strongest = sorted(kp_a, key=lambda k: -k.size)[:60]
rich = cv.drawKeypoints(frame_a, strongest, None, color=(0, 255, 255), flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
k0 = max(kp_a, key=lambda k: k.response if 15 < k.size < 30 else 0)
i0 = kp_a.index(k0)
fig, axes = plt.subplots(1, 3, figsize=(19, 4), gridspec_kw={"width_ratios": [1.4, 0.7, 1.4]})
axes[0].imshow(rich)
axes[0].set_title("60 самых крупных точек: круг это масштаб, отрезок направление")
axes[0].axis("off")
x0, y0, r0 = int(k0.pt[0]), int(k0.pt[1]), int(k0.size * 1.5)
axes[1].imshow(frame_a[max(y0 - r0, 0) : y0 + r0, max(x0 - r0, 0) : x0 + r0])
axes[1].set_title(f"одна точка, масштаб {k0.size:.0f}, угол {k0.angle:.0f}")
axes[1].axis("off")
axes[2].bar(range(128), des_a[i0], color=plt.cm.tab20(np.repeat(np.arange(16), 8) % 20))
axes[2].set_title("её дескриптор: 16 ячеек по 8 направлений")
axes[2].set_xlabel("номер числа в дескрипторе")
plt.tight_layout()
plt.show()

**Что видно:** точки стоят на пятнах, концах жилок и пересечениях, а не на ровных участках, и разного размера: мелкие на белых точках, крупные на пятнах крыльев. Дескриптор это 16 маленьких гистограмм подряд, цветом выделена каждая ячейка сетки.

## Сопоставление и тест Лоу

Для каждой точки B ищем ближайший дескриптор в A. Но у точки на повторяющемся узоре, например на одной из многих одинаковых белых точек, ближайший сосед может оказаться чужим. Тест Лоу: берём два ближайших и смотрим на отношение расстояний. Если лучший заметно ближе второго, соответствие надёжное. Если почти одинаковы, точка неоднозначна. Мы знаем настоящую гомографию, поэтому для каждого соответствия можем сказать, верное ли оно: ошибка после переноса меньше 3 пикселей.

In [ ]:
pairs = cv.BFMatcher(cv.NORM_L2).knnMatch(des_b, des_a, k=2)
pts_b = np.float32([kp_b[m.queryIdx].pt for m, _ in pairs])
pts_a = np.float32([kp_a[m.trainIdx].pt for m, _ in pairs])
ratio = np.array([m.distance / n.distance for m, n in pairs])
err = np.linalg.norm(cv.perspectiveTransform(pts_b[None], H_true)[0] - pts_a, axis=1)
correct = err < 3
print(f"всего соответствий {len(pairs)}, верных {correct.sum()}, доля {correct.mean():.2f}")
for r in [0.6, 0.7, 0.75, 0.8, 0.9]:
    sel = ratio < r
    print(f"ratio < {r}: осталось {sel.sum():4d}, верных среди них {correct[sel].mean():.2f}, потеряли верных {(~sel & correct).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(15, 3.6), gridspec_kw={"width_ratios": [1, 2]})
axes[0].hist(ratio[correct], bins=40, range=(0, 1), alpha=0.6, label="верные")
axes[0].hist(ratio[~correct], bins=40, range=(0, 1), alpha=0.6, label="неверные")
axes[0].axvline(0.75, c="k", ls="--")
axes[0].set_xlabel("d1 / d2")
axes[0].legend()

def draw_matches(ax, sel, title):
    canvas = np.hstack([frame_a, frame_b])
    ax.imshow(canvas)
    for i in np.flatnonzero(sel)[::3]:
        ax.plot([pts_a[i, 0], pts_b[i, 0] + FW], [pts_a[i, 1], pts_b[i, 1]], c="lime" if correct[i] else "red", lw=0.6)
    ax.set_title(title)
    ax.axis("off")

draw_matches(axes[1], ratio < 0.75, "после теста Лоу 0.75, каждое третье: зелёные верные, красные неверные")
plt.tight_layout()
plt.show()

**Что видно:** без теста верна только четверть соответствий из 1070: больше половины точек B лежит вне общей части кадров, и для них верного соседа просто нет, но ближайший всё равно находится. Распределения отношения у верных и неверных перекрываются мало: верные слева, неверные кучей у единицы. Порог 0.75 оставляет 279 соответствий, из них 88% верных, и теряет 33 верных из 278. Порог 0.6 даёт 99% верных, но теряет уже 54. Красные линии всё же остаются, и с ними разбирается следующий шаг.

## RANSAC: гомография несмотря на выбросы

Гомографию можно посчитать по 4 соответствиям. Если подставить все соответствия в метод наименьших квадратов, даже несколько неверных утащат ответ. RANSAC много раз берёт 4 случайных соответствия, считает гомографию и смотрит, сколько остальных с ней согласны. Побеждает гомография с самым большим числом согласных, и по ним её уточняют. Качество меряем ошибкой в углах кадра B против настоящей гомографии.

In [ ]:
sel = ratio < 0.75
H_ls, _ = cv.findHomography(pts_b[sel], pts_a[sel], 0)
H_ransac, inl = cv.findHomography(pts_b[sel], pts_a[sel], cv.RANSAC, 3.0)
inl = inl.ravel().astype(bool)

def corner_error(Hx):
    return np.linalg.norm(cv.perspectiveTransform(corners_b, Hx)[0] - true_corners, axis=1).max()

print(f"наименьшие квадраты по всем {sel.sum()}: ошибка в углах {corner_error(H_ls):.1f} px")
print(f"RANSAC: согласных {inl.sum()} из {sel.sum()}, ошибка в углах {corner_error(H_ransac):.2f} px")
print(f"среди согласных верных {correct[sel][inl].mean():.3f}, среди отброшенных {correct[sel][~inl].mean():.3f}")

fig, ax = plt.subplots(figsize=(10, 4.2))
ax.imshow(frame_a, extent=(0, FW, Hp, 0))
for Hx, c, lw, ls, name in [(H_ls, "red", 2, "-", "наименьшие квадраты"), (H_ransac, "lime", 4, "-", "RANSAC"), (H_true, "black", 1.5, "--", "настоящая")]:
    q = cv.perspectiveTransform(corners_b, Hx)[0]
    ax.add_patch(plt.Polygon(q, fill=False, ec=c, lw=lw, ls=ls, label=name))
ax.set_xlim(-50, 1.7 * FW)
ax.set_ylim(Hp + 60, -60)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
ax.set_title("где окажется кадр B в координатах кадра A")
plt.show()

**Что видно:** нескольких неверных соответствий хватает, чтобы наименьшие квадраты дали рамку не там. RANSAC находит гомографию с ошибкой около пикселя в углах, и его рамка совпадает с настоящей. Отброшенные RANSAC соответствия почти все неверные.

## SIFT против ORB при повороте и масштабе

ORB это быстрая замена SIFT: углы FAST и двоичный дескриптор, который сравнивают по числу различающихся битов. Повернём и уменьшим кадр A известным преобразованием и посчитаем, сколько верных соответствий остаётся после теста Лоу.

In [ ]:
orb = cv.ORB_create(2000)
detectors = {"SIFT": (sift, cv.NORM_L2), "ORB": (orb, cv.NORM_HAMMING)}

def correct_matches(detector, norm, img, M):
    warped = cv.warpAffine(img, M, (img.shape[1], img.shape[0]))
    k1, d1 = detector.detectAndCompute(img, None)
    k2, d2 = detector.detectAndCompute(warped, None)
    good = [m for m, n in cv.BFMatcher(norm).knnMatch(d1, d2, k=2) if m.distance < 0.75 * n.distance]
    p1 = np.float32([k1[m.queryIdx].pt for m in good])
    p2 = np.float32([k2[m.trainIdx].pt for m in good])
    return int((np.linalg.norm(p1 @ M[:, :2].T + M[:, 2] - p2, axis=1) < 3).sum())

center = (FW / 2, Hp / 2)
angles = [0, 15, 30, 45, 60, 90]
scales = [1.0, 0.8, 0.6, 0.45, 0.3]
fig, axes = plt.subplots(1, 3, figsize=(17, 3.6), gridspec_kw={"width_ratios": [1, 1, 0.8]})
for name, (det, norm) in detectors.items():
    by_angle = [correct_matches(det, norm, gray_a, cv.getRotationMatrix2D(center, a, 1.0)) for a in angles]
    by_scale = [correct_matches(det, norm, gray_a, cv.getRotationMatrix2D(center, 0, s_)) for s_ in scales]
    axes[0].plot(angles, by_angle, "o-", label=name)
    axes[1].plot(scales, by_scale, "o-", label=name)
    print(name, "по углу:", by_angle, " по масштабу:", by_scale)
axes[0].set_xlabel("поворот, градусы")
axes[1].set_xlabel("масштаб")
axes[1].invert_xaxis()
for ax in axes[:2]:
    ax.set_ylabel("верных соответствий")
    ax.legend()
axes[2].imshow(cv.warpAffine(gray_a, cv.getRotationMatrix2D(center, 45, 0.45), (FW, Hp)), cmap="gray")
axes[2].set_title("пример: поворот 45, масштаб 0.45")
axes[2].axis("off")
plt.tight_layout()
plt.show()

**Что видно:** точка при угле 0 и масштабе 1 это та же картинка, там совпадает всё. После любого поворота у обоих методов остаётся примерно одинаковое число соответствий, что на 15 градусов, что на 90: оба поворачивают окно по направлению точки. ORB мы попросили найти 2000 точек, и соответствий у него больше. Но при уменьшении он теряет их быстрее: от 1963 до 77 к масштабу 0.3, а SIFT от 1172 до 89, у SIFT более тонкая пирамида масштабов. SIFT обычно точнее и устойчивее, ORB во много раз быстрее, и в телефоне и на роботе часто выбирают его.

## Склейка

Переносим кадр B в координаты A найденной гомографией и кладём на общий холст. Простая склейка: где есть A, берём A, остальное из B. Сразу видно две проблемы. Шов: B светлее, потому что у камеры была другая экспозиция. Поправим её: в общей части посчитаем, во сколько раз B ярче, и поделим на это число. И резкая граница: вместо резкого перехода смешаем кадры с весами, которые плавно спадают к краю каждого кадра, это feathering.

In [ ]:
top = 70
canvas_w, canvas_h = int(true_corners[:, 0].max()) + 10, Hp + 2 * top
shift = np.array([[1, 0, 0], [0, 1, top], [0, 0, 1]], float)
warp_a = cv.warpPerspective(frame_a, shift, (canvas_w, canvas_h)).astype(np.float32)
warp_b = cv.warpPerspective(frame_b, shift @ H_ransac, (canvas_w, canvas_h)).astype(np.float32)
mask_a = cv.warpPerspective(np.full(frame_a.shape[:2], 255, np.uint8), shift, (canvas_w, canvas_h), flags=cv.INTER_NEAREST)
mask_b = cv.warpPerspective(np.full(frame_b.shape[:2], 255, np.uint8), shift @ H_ransac, (canvas_w, canvas_h), flags=cv.INTER_NEAREST)
mask_b = cv.erode(mask_b, np.ones((3, 3), np.uint8))
both = (mask_a > 0) & (mask_b > 0)

naive = np.where(mask_a[..., None] > 0, warp_a, warp_b)
gain = warp_a[both].mean(0) / warp_b[both].mean(0)
print("во сколько раз A темнее B в общей части, по каналам:", gain.round(3))
warp_b_gain = np.clip(warp_b * gain, 0, 255)
with_gain = np.where(mask_a[..., None] > 0, warp_a, warp_b_gain)
wa = cv.distanceTransform(mask_a, cv.DIST_L2, 5)
wb = cv.distanceTransform(mask_b, cv.DIST_L2, 5)
feather = (warp_a * wa[..., None] + warp_b_gain * wb[..., None]) / np.maximum(wa + wb, 1e-6)[..., None]

seam_x = FW - 1
fig, axes = plt.subplots(2, 3, figsize=(20, 8), gridspec_kw={"height_ratios": [1.3, 1]})
for j, (im, title) in enumerate([(naive, "просто сверху"), (with_gain, "с поправкой экспозиции"), (feather, "поправка + feathering")]):
    axes[0, j].imshow(im.astype(np.uint8))
    axes[0, j].add_patch(plt.Rectangle((seam_x - 70, top + 140), 140, 140, fill=False, ec="cyan", lw=1.5))
    axes[0, j].set_title(title)
    axes[1, j].imshow(im[top + 140 : top + 280, seam_x - 70 : seam_x + 70].astype(np.uint8), interpolation="nearest")
    axes[1, j].set_title("увеличено у правого края кадра A")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

panorama = feather.astype(np.uint8)
fig, axes = plt.subplots(1, 3, figsize=(20, 3.6), gridspec_kw={"width_ratios": [1, 1, 1.9]})
axes[0].imshow(frame_a)
axes[0].set_title("кадр A")
axes[1].imshow(frame_b)
axes[1].set_title("кадр B")
axes[2].imshow(panorama)
axes[2].set_title("панорама")
for ax in axes:
    ax.axis("off")
plt.show()

**Что видно:** при простой склейке по правому краю кадра A идёт резкая граница яркости, и даже если геометрия точная, глаз видит шов. После поправки экспозиции шов почти исчез, но на самых светлых участках переход ещё виден: там B упёрся в 255, и деление на одно число затемнило его неверно. Поправка одна на весь кадр и ничего не знает о насыщении. Feathering смешивает кадры в общей части с весами по расстоянию до края кадра, и переход становится плавным. Геометрический шов не виден вовсе: гомография найдена с точностью около пикселя.

В настоящих панорамах ещё много шагов: кадров много, и гомографии уточняют сразу для всех; камеру поворачивают, и кадры проецируют на цилиндр или сферу; шов прокладывают там, где кадры меньше всего различаются; смешивают по частотам. Всё это есть в `cv.Stitcher`. Но основа та же: особые точки, тест Лоу, RANSAC, гомография.

## Источники

[OpenCV: Smoothing Images](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html),
[Canny Edge Detection](https://docs.opencv.org/4.x/da/d22/tutorial_py_canny.html),
[Image Thresholding](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html),
[Morphological Transformations](https://docs.opencv.org/4.x/d9/d61/tutorial_py_morphological_ops.html),
[Lowe. Distinctive Image Features from Scale-Invariant Keypoints. IJCV 2004](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf),
[Rublee et al. ORB: an efficient alternative to SIFT or SURF. ICCV 2011](https://ieeexplore.ieee.org/document/6126544),
[Fischler, Bolles. Random Sample Consensus. 1981](https://dl.acm.org/doi/10.1145/358669.358692),
[Brown, Lowe. Automatic Panoramic Image Stitching using Invariant Features. IJCV 2007](https://link.springer.com/article/10.1007/s11263-006-0002-3),
[OpenCV: Feature Matching + Homography](https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html).
Ядра Собеля из [семинара по свёрткам](https://github.com/markblumenau/dl-course-hse-masters/blob/main/seminars/03_Image_Convolution.ipynb) курса DL.